# Question 3 - VAE Loss Function
Total loss = Reconstruction loss (BCE, summed over the 28x28 pixels) + $\beta$ x KL divergence

$$KL = -\frac{1}{2}\sum\left(1 + \log\sigma^2 - \mu^2 - \sigma^2\right)$$

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
def vae_loss(x, x_hat, mu, log_var, kl_weight=1.0):
    eps = 1e-7
    x_hat = tf.clip_by_value(x_hat, eps, 1 - eps)
    # reconstruction loss: BCE summed over all 28x28 pixels
    recon = -tf.reduce_sum(x * tf.math.log(x_hat) + (1 - x) * tf.math.log(1 - x_hat), axis=[1, 2])
    # KL divergence between N(mu, sigma^2) and N(0, I)
    kl = -0.5 * tf.reduce_sum(1 + log_var - tf.square(mu) - tf.exp(log_var), axis=1)
    return tf.reduce_mean(recon + kl_weight * kl), tf.reduce_mean(recon), tf.reduce_mean(kl)

### Demo: train a small VAE with KL weight = 1 and KL weight = 0

In [ ]:
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
train_ds = tf.data.Dataset.from_tensor_slices(x_train).shuffle(10000).batch(128)

In [ ]:
class VAE(keras.Model):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = keras.Sequential([layers.Flatten(), layers.Dense(256, activation='relu')])
        self.z_mean = layers.Dense(latent_dim)
        self.z_log_var = layers.Dense(latent_dim)
        self.decoder = keras.Sequential([layers.Dense(256, activation='relu'),
                                         layers.Dense(784, activation='sigmoid'),
                                         layers.Reshape((28, 28))])

    def call(self, x):
        h = self.encoder(x)
        mu, log_var = self.z_mean(h), self.z_log_var(h)
        z = mu + tf.exp(0.5 * log_var) * tf.random.normal(tf.shape(mu))
        return self.decoder(z), mu, log_var


def train_vae(kl_weight, epochs=3):
    model = VAE()
    opt = keras.optimizers.Adam(1e-3)

    @tf.function
    def step(x):
        with tf.GradientTape() as tape:
            x_hat, mu, log_var = model(x)
            loss, recon, kl = vae_loss(x, x_hat, mu, log_var, kl_weight)
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss, recon, kl

    for ep in range(epochs):
        for x in train_ds:
            loss, recon, kl = step(x)
        print(f"kl_weight={kl_weight} epoch {ep+1}: loss={loss:.2f} recon={recon:.2f} kl={kl:.2f}")
    return model

vae_kl1 = train_vae(kl_weight=1.0)
vae_kl0 = train_vae(kl_weight=0.0)

In [ ]:
# Evaluate on test set
for name, m, w in [("KL weight 1", vae_kl1, 1.0), ("KL weight 0", vae_kl0, 0.0)]:
    x_hat, mu, lv = m(x_test[:1000])
    loss, recon, kl = vae_loss(x_test[:1000], x_hat, mu, lv, w)
    print(f"{name}: test recon={recon:.2f}, test KL={kl:.2f}")

# Generate samples from z ~ N(0, I)
z = tf.random.normal((10, 2))
fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for row, (name, m) in enumerate([("KL=1", vae_kl1), ("KL=0", vae_kl0)]):
    imgs = m.decoder(z).numpy()
    for i in range(10):
        axes[row, i].imshow(imgs[i], cmap='gray'); axes[row, i].axis('off')
    axes[row, 0].set_title(name, loc='left')
plt.show()

### 2. What happens if the KL weight is 0?
The model becomes a **plain autoencoder**. Nothing pushes the latent codes toward $\mathcal{N}(0, I)$, so the latent space is irregular and has gaps, and the encoder variances can shrink toward zero.
- Reconstructions stay good, often sharper.
- **New samples drawn from $\mathcal{N}(0, I)$ look poor**: blurry, distorted or not digit-like, because those z values may fall in regions the decoder never saw. Smooth interpolation between codes is also lost.